### Digital [Binary] Signal Detection

### Symbol Detection in Additive White Gaussian Noise


$\large {\textrm{Tx: } \{p(a_n)\}: {A} }\longrightarrow$
$ {\textrm{Cx: }y = {A} +\eta :\mathcal{N}_{\eta}(0,\sigma_{\eta}) }\longrightarrow $
${\textrm{Rx: } }$

$\operatorname{M}({A}, {\hat{A}})\Rightarrow\{p(y|a_n)\}$

- metrics for comparison
$A = \{a_n{\in }\mathbb{R}:n {\in} N\}$ -- Sets representing a finite number of indexed objects/states or relationships (symbols)



In [ ]:
# ============================================================
# Cell 1 — Imports and plotting configuration
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from matplotlib import colors
from matplotlib import rcParams
from scipy import stats

# Global Matplotlib configuration
rcParams["figure.figsize"] = (12, 6)
rcParams["figure.dpi"] = 140
rcParams["axes.labelsize"] = 13
rcParams["axes.titlesize"] = 14
rcParams["legend.fontsize"] = 11
rcParams["lines.markersize"] = 7
rcParams["axes.grid"] = True

#### PDF Estimation

- Histogram method
- Kernel Density Estimation (KDE)  

In [ ]:
# ============================================================
# Cell 2 — Colored density histogram utility
# ============================================================

def colored_histogram(
    values: np.ndarray,
    axis: plt.Axes,
    *,
    bins: str | int = "fd",
    alpha: float = 0.81,
) -> None:
    """
    Plot a density-normalized histogram whose bar colors represent
    the relative frequency of the corresponding bins.

    Parameters
    ----------
    values : array-like
        Numerical samples to be represented by the histogram.

    axis : matplotlib.axes.Axes
        Axis object on which the histogram is drawn.

    bins : str or int, optional
        Histogram bin-selection method or number of bins.
        The default value, "fd", applies the Freedman-Diaconis rule.

    alpha : float, optional
        Transparency of the histogram bars. Must lie in [0, 1].
    """
    values = np.asarray(values, dtype=float).reshape(-1)

    if values.size == 0:
        raise ValueError("The input array must not be empty.")

    if not np.all(np.isfinite(values)):
        raise ValueError("The input array must contain only finite values.")

    if not 0.0 <= alpha <= 1.0:
        raise ValueError("The alpha parameter must lie in the interval [0, 1].")

    densities, _, patches = axis.hist(
        values,
        bins=bins,
        density=True,
        alpha=alpha,
        edgecolor="black",
        linewidth=0.5,
    )

    if not np.all(np.isfinite(densities)):
        raise ValueError(
            "The histogram produced non-finite density values. "
            "Check the input samples or bin configuration."
        )

    max_density = np.max(densities)

    if max_density > 0:
        relative_densities = densities / max_density
    else:
        relative_densities = np.zeros_like(densities)

    color_normalizer = colors.Normalize(vmin=0.0, vmax=1.0)
    colormap = plt.colormaps["viridis"]

    for relative_density, patch in zip(relative_densities, patches):
        patch.set_facecolor(colormap(color_normalizer(relative_density)))

    axis.set_ylabel("Estimated probability density")
    axis.grid(True, alpha=0.3)

In [ ]:
# ============================================================
# Cell 3 — One-dimensional kernel-density estimation
# ============================================================

def estimate_kde(
    samples: np.ndarray,
    evaluation_points: np.ndarray | None = None,
    *,
    method: str = "silverman",
    resolution: int = 500,
    margin_std: float = 3.0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Estimate a one-dimensional probability density using a Gaussian kernel.

    This utility is intended only for empirical visualization. For the
    analytical binary-detection model, use the exact Gaussian PDFs.
    """
    samples = np.asarray(samples, dtype=float).reshape(-1)

    if samples.size < 2:
        raise ValueError("At least two samples are required for KDE.")

    if not np.all(np.isfinite(samples)):
        raise ValueError("The samples must contain only finite values.")

    sample_std = np.std(samples, ddof=1)

    if np.isclose(sample_std, 0.0):
        raise ValueError("KDE cannot be estimated from constant-valued samples.")

    if not isinstance(resolution, (int, np.integer)) or resolution < 2:
        raise ValueError("The resolution must be an integer greater than or equal to 2.")

    if not np.isfinite(margin_std) or margin_std < 0:
        raise ValueError("The margin_std parameter must be finite and non-negative.")

    if not isinstance(method, str):
        raise TypeError("The KDE method must be a string.")

    method = method.lower().strip().replace(" ", "_").replace("-", "_")

    if evaluation_points is None:
        lower_limit = np.min(samples) - margin_std * sample_std
        upper_limit = np.max(samples) + margin_std * sample_std

        evaluation_points = np.linspace(
            lower_limit,
            upper_limit,
            resolution,
        )
    else:
        evaluation_points = np.asarray(
            evaluation_points,
            dtype=float,
        ).reshape(-1)

        if evaluation_points.size == 0:
            raise ValueError("The evaluation grid must not be empty.")

        if not np.all(np.isfinite(evaluation_points)):
            raise ValueError(
                "The evaluation grid must contain only finite values."
            )

    if method in {"silverman", "scott"}:
        kde = stats.gaussian_kde(
            samples,
            bw_method=method,
        )

        estimated_pdf = kde(evaluation_points)

    elif method == "cross_validation":
        from sklearn.model_selection import GridSearchCV
        from sklearn.neighbors import KernelDensity

        candidate_bandwidths = np.geomspace(
            0.05 * sample_std,
            2.0 * sample_std,
            40,
        )

        cv_folds = min(5, samples.size)

        grid_search = GridSearchCV(
            estimator=KernelDensity(kernel="gaussian"),
            param_grid={"bandwidth": candidate_bandwidths},
            cv=cv_folds,
            n_jobs=-1,
        )

        grid_search.fit(samples[:, np.newaxis])

        optimal_kde = grid_search.best_estimator_

        log_density = optimal_kde.score_samples(
            evaluation_points[:, np.newaxis]
        )

        estimated_pdf = np.exp(log_density)

    else:
        raise ValueError(
            "Unsupported KDE method. Select 'silverman', 'scott', "
            "or 'cross_validation'."
        )

    if not np.all(np.isfinite(estimated_pdf)):
        raise ValueError("The KDE produced non-finite density values.")

    return evaluation_points, estimated_pdf


#### 1. Binary source and channel model ($N=2$)

Consider a binary source that transmits one of two real-valued symbols:
$A \in \{a_0,a_1\},\qquad a_0<a_1.$

The prior probabilities are:

$
P(H_0)=P(A=a_0)=p_0,
\qquad
P(H_1)=P(A=a_1)=p_1,
$

with:

$
p_0+p_1=1.
$

The receiver observes a noisy scalar sample:

$
Y=A+\eta,
$

where the additive noise is Gaussian:

$
\eta\sim\mathcal{N}(0,\sigma_\eta^2).
$

Therefore, the binary hypotheses are:

$
\begin{aligned}
H_0 &: \quad Y=a_0+\eta,\\[4pt]
H_1 &: \quad Y=a_1+\eta.
\end{aligned}
$

In [ ]:
# ============================================================
# Cell 4 — Binary-symbol generation and waveform construction
# ============================================================

def generate_symbols(
    num_symbols: int,
    symbol_values: np.ndarray,
    symbol_probabilities: np.ndarray,
    *,
    rng: np.random.Generator,
) -> np.ndarray:
    """
    Generate a random binary-symbol sequence according to specified
    prior probabilities.

    Parameters
    ----------
    num_symbols : int
        Number of symbols to generate.

    symbol_values : array-like
        Two distinct symbol amplitudes:
        [a0, a1], with a0 < a1.

    symbol_probabilities : array-like
        Prior probabilities:
        [P(H0), P(H1)].
        Both values must be strictly positive and their sum must equal one.

    rng : np.random.Generator
        Explicit random-number generator used for reproducibility.

    Returns
    -------
    np.ndarray
        Generated binary-symbol sequence.
    """
    symbol_values = np.asarray(
        symbol_values,
        dtype=float,
    ).reshape(-1)

    symbol_probabilities = np.asarray(
        symbol_probabilities,
        dtype=float,
    ).reshape(-1)

    if not isinstance(num_symbols, (int, np.integer)) or num_symbols <= 0:
        raise ValueError("num_symbols must be a positive integer.")

    if symbol_values.size != 2:
        raise ValueError(
            "Binary detection requires exactly two symbol amplitudes."
        )

    if not np.all(np.isfinite(symbol_values)):
        raise ValueError(
            "The symbol amplitudes must contain only finite values."
        )

    if not symbol_values[0] < symbol_values[1]:
        raise ValueError(
            "The symbol amplitudes must satisfy symbol_values[0] < symbol_values[1]."
        )

    if symbol_probabilities.size != 2:
        raise ValueError(
            "Provide exactly two prior probabilities: [P(H0), P(H1)]."
        )

    if not np.all(np.isfinite(symbol_probabilities)):
        raise ValueError(
            "The prior probabilities must contain only finite values."
        )

    if np.any(symbol_probabilities <= 0):
        raise ValueError(
            "The prior probabilities must be strictly positive."
        )

    if not np.isclose(np.sum(symbol_probabilities), 1.0):
        raise ValueError(
            "The prior probabilities must sum to one."
        )

    if not isinstance(rng, np.random.Generator):
        raise TypeError(
            "rng must be an instance of np.random.Generator."
        )

    return rng.choice(
        symbol_values,
        size=num_symbols,
        p=symbol_probabilities,
    )


def generate_rectangular_waveform(
    symbols: np.ndarray,
    samples_per_symbol: int,
) -> np.ndarray:
    """
    Convert a discrete symbol sequence into a sampled rectangular-pulse
    waveform by repeating each symbol amplitude throughout one symbol
    interval.

    Parameters
    ----------
    symbols : array-like
        Sequence of transmitted symbol amplitudes.

    samples_per_symbol : int
        Number of samples used to represent each symbol interval.

    Returns
    -------
    np.ndarray
        Sampled rectangular-pulse waveform.
    """
    symbols = np.asarray(symbols, dtype=float).reshape(-1)

    if symbols.size == 0:
        raise ValueError("The symbol sequence must not be empty.")

    if not np.all(np.isfinite(symbols)):
        raise ValueError(
            "The symbol sequence must contain only finite values."
        )

    if (
        not isinstance(samples_per_symbol, (int, np.integer))
        or samples_per_symbol <= 0
    ):
        raise ValueError(
            "samples_per_symbol must be a positive integer."
        )

    return np.repeat(
        symbols,
        samples_per_symbol,
    )

In [ ]:
# ============================================================
# Cell 5 — Simulation parameters and transmitted waveform
# ============================================================

# Binary source configuration
num_symbols = 4096
symbol_values = np.array([-1.0, 1.0])
symbol_probabilities = np.array([0.5, 0.5])

# Rectangular-pulse waveform configuration
samples_per_symbol = 32
expected_num_samples = num_symbols * samples_per_symbol

# Reproducibility
symbol_seed = 42
symbol_rng = np.random.default_rng(seed=symbol_seed)

# Generate the discrete binary-symbol sequence
symbols = generate_symbols(
    num_symbols=num_symbols,
    symbol_values=symbol_values,
    symbol_probabilities=symbol_probabilities,
    rng=symbol_rng,
)

# Generate the sampled rectangular-pulse waveform
transmitted_waveform = generate_rectangular_waveform(
    symbols=symbols,
    samples_per_symbol=samples_per_symbol,
)

# Total number of samples in the transmitted waveform
num_samples = transmitted_waveform.size

# Integer sample positions: 0, 1, 2, ..., num_samples - 1
sample_indices = np.arange(num_samples)

# Normalized time: one unit corresponds to one symbol duration
normalized_time = sample_indices / samples_per_symbol

# Consistency checks
assert symbols.size == num_symbols
assert transmitted_waveform.size == expected_num_samples
assert sample_indices.size == num_samples
assert normalized_time.size == num_samples

print(f"Number of symbols:       {num_symbols}")
print(f"Samples per symbol:      {samples_per_symbol}")
print(f"Total waveform samples:  {num_samples}")
print(f"Empirical P(H0):         {np.mean(symbols == symbol_values[0]):.4f}")
print(f"Empirical P(H1):         {np.mean(symbols == symbol_values[1]):.4f}")

#### Cx
Channel simulation: Additive White Gaussian Noise: $\eta(t),$ with $\mathbb{E}\{\eta(t)\}=0$

WGN:
1.  **Additive:** It's simply added to our signal, $y = \text{signal}(a_n) + \eta$.
2.  **White:** It has a constant power spectral density across all frequencies, meaning it affects all frequencies equally.
3.  **Gaussian:** Its amplitude follows a Gaussian (normal) probability distribution, with a mean (expected value) of zero, $\mathbb{E}\{\eta(t)\} = 0$. The variance of the noise ($\sigma_{\eta}^2$) determines its power.


In [ ]:
# ============================================================
# Cell 6 — AWGN generation and empirical characterization
# ============================================================

# ------------------------------------------------------------
# Noise configuration
# ------------------------------------------------------------

noise_mean = 0.0
noise_variance = 3.0

if noise_variance <= 0:
    raise ValueError("noise_variance must be strictly positive.")

noise_std = np.sqrt(noise_variance)

# Use a dedicated generator so that the AWGN realization is
# reproducible and independent of the transmitted-symbol sequence.
noise_seed = 123
noise_rng = np.random.default_rng(seed=noise_seed)

# ------------------------------------------------------------
# Generate the AWGN samples
# ------------------------------------------------------------

noise = noise_rng.normal(
    loc=noise_mean,
    scale=noise_std,
    size=num_samples,
)

if not np.all(np.isfinite(noise)):
    raise ValueError("The generated noise contains non-finite values.")

# ------------------------------------------------------------
# Construct the received waveform
# ------------------------------------------------------------

received_waveform = transmitted_waveform + noise

# ------------------------------------------------------------
# Numerical consistency checks
# ------------------------------------------------------------

assert noise.size == num_samples
assert sample_indices.size == num_samples
assert received_waveform.size == transmitted_waveform.size

estimated_noise_mean = np.mean(noise)
estimated_noise_variance = np.var(noise, ddof=1)

# ------------------------------------------------------------
# 1. Empirical and exact amplitude distributions
# ------------------------------------------------------------

evaluation_points, noise_pdf_kde = estimate_kde(
    samples=noise,
    method="silverman",
    resolution=600,
)

noise_pdf_exact = stats.norm.pdf(
    evaluation_points,
    loc=noise_mean,
    scale=noise_std,
)

# ------------------------------------------------------------
# 2. Normalized sample autocorrelation
# ------------------------------------------------------------

max_lag = min(60, noise.size - 1)
centered_noise = noise - np.mean(noise)

autocorrelation = np.array([
    np.dot(centered_noise[:noise.size-lag], centered_noise[lag:])
    for lag in range(max_lag + 1)
])

normalized_autocorrelation = autocorrelation / autocorrelation[0]
lags = np.arange(max_lag + 1)

# Approximate 95% reference limits for nonzero-lag autocorrelation values
confidence_limit = 1.96 / np.sqrt(noise.size)

# ------------------------------------------------------------
# 3. Visualization
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(12, 11),
    constrained_layout=True,
)

# Time-domain realization
axes[0].plot(
    sample_indices,
    noise,
    linewidth=0.8,
    label="AWGN realization",
)

axes[0].set_title("AWGN time-domain realization")
axes[0].set_xlabel("Sample index")
axes[0].set_ylabel("Noise amplitude")
axes[0].set_xlim(0, num_samples - 1)
axes[0].legend()

# Amplitude distribution
colored_histogram(
    values=noise,
    axis=axes[1],
)

axes[1].plot(
    evaluation_points,
    noise_pdf_kde,
    linewidth=2.0,
    label="Empirical KDE",
)

axes[1].plot(
    evaluation_points,
    noise_pdf_exact,
    linestyle="--",
    linewidth=2.0,
    label="Exact Gaussian PDF",
)

axes[1].set_title("AWGN amplitude distribution")
axes[1].set_xlabel("Noise amplitude")
axes[1].set_ylabel("Probability density")
axes[1].set_xlim(symbol_values[0]-2*noise_std, symbol_values[1]+2*noise_std)
axes[1].legend()

# Normalized autocorrelation
axes[2].stem(
    lags,
    normalized_autocorrelation,
    basefmt=" ",
)

axes[2].axhline(
    confidence_limit,
    linestyle="--",
    linewidth=1.5,
    label=rf"Approximate 95% limits: $\pm {confidence_limit:.3f}$",
)

axes[2].axhline(
    -confidence_limit,
    linestyle="--",
    linewidth=1.5,
)

axes[2].axhline(
    0.0,
    linewidth=1.0,
)

axes[2].set_title("Normalized sample autocorrelation of the AWGN realization")
axes[2].set_xlabel("Lag")
axes[2].set_ylabel("Normalized autocorrelation")
axes[2].set_xlim(0, max_lag)
axes[2].legend()

plt.show()

# ------------------------------------------------------------
# 4. Numerical summary
# ------------------------------------------------------------

nonzero_lag_autocorrelation = normalized_autocorrelation[1:]

num_lags_outside_limits = np.count_nonzero(
    np.abs(nonzero_lag_autocorrelation) > confidence_limit
)

maximum_nonzero_lag_acf = np.max(
    np.abs(nonzero_lag_autocorrelation)
)


print(f"Target noise mean:                 {noise_mean:.6f}")
print(f"Estimated noise mean:              {estimated_noise_mean:.6f}")
print(f"Target noise variance:             {noise_variance:.6f}")
print(f"Estimated noise variance:          {estimated_noise_variance:.6f}")
print(
    f"Maximum absolute nonzero-lag ACF:  "
    f"{maximum_nonzero_lag_acf:.6f}"
)
print(f"Reference confidence limit:        {confidence_limit:.6f}")
print(f"Lags outside reference limits:     {num_lags_outside_limits} of {max_lag}")

### Received Signal ($y$) in an Additive Gaussian Channel

The mathematical representation of the received signal, $y(t)$, in an Additive Gaussian Channel is given by:

$y(t) = \sum_{\forall k}{{a}^{(k)}_n}\operatorname{rect}_{\delta t}(t-k\delta t) +\eta(t)$

Here's what each part means:
*   $\sum_{\forall k}{{a}^{(k)}_n}\operatorname{rect}_{\delta t}(t-k\delta t)$: This term represents our original transmitted digital signal. It's a sum of rectangular pulses, where each pulse corresponds to a  symbol ${a}^{(k)}_n$, transmitted within $k\delta t$, and has a duration $\delta t$.
*   $\eta(t)$: This is the Additive White Gaussian Noise (AWGN) introduced by the channel.

The corresponding conditional probability-density functions are:

$
p(y\mid H_0)
=
\frac{1}{\sqrt{2\pi\sigma_\eta^2}}
\exp\!\left[
-\frac{(y-a_0)^2}{2\sigma_\eta^2}
\right],
$

and:

$
p(y\mid H_1)
=
\frac{1}{\sqrt{2\pi\sigma_\eta^2}}
\exp\!\left[
-\frac{(y-a_1)^2}{2\sigma_\eta^2}
\right].
$

In [ ]:
# ============================================================
# Cell 7 — Received waveform and prior-weighted likelihoods
# ============================================================

# Binary symbol amplitudes and prior probabilities
a0, a1 = symbol_values
prior_h0, prior_h1 = symbol_probabilities

if a0 >= a1:
    raise ValueError("The binary amplitudes must satisfy a0 < a1.")

if prior_h0 <= 0 or prior_h1 <= 0:
    raise ValueError("Both prior probabilities must be strictly positive.")

if not np.isclose(prior_h0 + prior_h1, 1.0):
    raise ValueError("The symbol probabilities must sum to one.")

assert received_waveform.size == transmitted_waveform.size
assert received_waveform.size == noise.size
assert sample_indices.size == num_samples

# ------------------------------------------------------------
# 1. Received sampled waveform
# ------------------------------------------------------------

num_visible_symbols = 24

num_visible_samples = min(
    num_samples,
    num_visible_symbols * samples_per_symbol,
)

visible_indices = sample_indices[:num_visible_samples]

# ------------------------------------------------------------
# 2. Scalar observations under each hypothesis
# ------------------------------------------------------------

# Conditional observation models:
# Y | H0 ~ N(a0, noise_variance)
# Y | H1 ~ N(a1, noise_variance)
#
# For KDE visualization, both conditional observation arrays are generated
# by shifting the same empirical AWGN realization. Therefore, the two KDEs
# are correlated shifted estimates of the same noise distribution, not two
# independent Monte Carlo simulations. This is intentional here because the
# KDE is used only as a visual approximation to the exact Gaussian PDFs.

observations_h0 = a0 + noise
observations_h1 = a1 + noise

# Evaluation grid with margins wide enough to display the Gaussian tails
pdf_margin_std = 4.0

lower_limit = min(a0, a1) - pdf_margin_std * noise_std
upper_limit = max(a0, a1) + pdf_margin_std * noise_std

evaluation_points = np.linspace(
    lower_limit,
    upper_limit,
    1000,
)

# ------------------------------------------------------------
# 3. Exact conditional PDFs
# ------------------------------------------------------------

pdf_h0_exact = stats.norm.pdf(
    evaluation_points,
    loc=a0,
    scale=noise_std,
)

pdf_h1_exact = stats.norm.pdf(
    evaluation_points,
    loc=a1,
    scale=noise_std,
)

# ------------------------------------------------------------
# 4. Prior-weighted exact likelihoods used by the MAP detector
# ------------------------------------------------------------

weighted_pdf_h0_exact = prior_h0 * pdf_h0_exact
weighted_pdf_h1_exact = prior_h1 * pdf_h1_exact

# ------------------------------------------------------------
# 5. Optional empirical KDE approximations
# ------------------------------------------------------------

_, pdf_h0_kde = estimate_kde(
    samples=observations_h0,
    evaluation_points=evaluation_points,
    method="silverman",
)

_, pdf_h1_kde = estimate_kde(
    samples=observations_h1,
    evaluation_points=evaluation_points,
    method="silverman",
)

weighted_pdf_h0_kde = prior_h0 * pdf_h0_kde
weighted_pdf_h1_kde = prior_h1 * pdf_h1_kde

# ------------------------------------------------------------
# 6. Visualization
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(12, 8),
    constrained_layout=True,
)

# Transmitted and received waveforms
axes[0].step(
    visible_indices,
    transmitted_waveform[:num_visible_samples],
    where="post",
    linewidth=2.0,
    label="Transmitted waveform",
)

axes[0].plot(
    visible_indices,
    received_waveform[:num_visible_samples],
    linewidth=1.0,
    alpha=0.65,
    label="Received waveform",
)

axes[0].set_title(
    "Binary rectangular-pulse waveform transmitted through an AWGN channel"
)

axes[0].set_xlabel("Sample index")
axes[0].set_ylabel("Amplitude")
axes[0].set_xlim(0, num_visible_samples - 1)
axes[0].set_ylim(symbol_values[0]-2*noise_std, symbol_values[1]+2*noise_std)
axes[0].legend()

# Prior-weighted exact likelihoods and empirical KDE approximations
axes[1].plot(
    evaluation_points,
    weighted_pdf_h0_exact,
    linewidth=2.5,
    label=r"Exact: $P(H_0)\,p(y \mid H_0)$",
)

axes[1].plot(
    evaluation_points,
    weighted_pdf_h1_exact,
    linewidth=2.5,
    label=r"Exact: $P(H_1)\,p(y \mid H_1)$",
)

axes[1].plot(
    evaluation_points,
    weighted_pdf_h0_kde,
    linestyle="--",
    linewidth=1.5,
    label=r"KDE: $P(H_0)\,\hat{p}(y \mid H_0)$",
)

axes[1].plot(
    evaluation_points,
    weighted_pdf_h1_kde,
    linestyle="--",
    linewidth=1.5,
    label=r"KDE: $P(H_1)\,\hat{p}(y \mid H_1)$",
)

axes[1].set_title(
    "Prior-weighted likelihoods for MAP binary detection"
)

axes[1].set_xlabel("Received-sample amplitude")
axes[1].set_ylabel("Weighted probability density")
axes[1].set_xlim(a0-2*noise_std, a1+2*noise_std)
axes[1].legend()

plt.show()

####  Likelihood-ratio derivation of the MAP detector

The maximum-a-posteriori (MAP) detector selects the hypothesis with the largest posterior probability:

$
P(H_1\mid y)
\mathop{\gtrless}_{H_0}^{H_1}
P(H_0\mid y).
$

Using Bayes' rule:

$
P(H_i\mid y)
=
\frac{p(y\mid H_i)P(H_i)}{p(y)},
$

the common denominator $p(y)$ can be removed from the comparison. Therefore:

$
p(y\mid H_1)P(H_1)
\mathop{\gtrless}_{H_0}^{H_1}
p(y\mid H_0)P(H_0).
$

Dividing both sides by $p(y\mid H_0)P(H_1)$ gives the likelihood-ratio test:

$
\Lambda(y)
=
\frac{p(y\mid H_1)}{p(y\mid H_0)}
\mathop{\gtrless}_{H_0}^{H_1}
\frac{P(H_0)}{P(H_1)}.
$

For the AWGN binary model:

$
p(y\mid H_i)
=
\frac{1}{\sqrt{2\pi\sigma_\eta^2}}
\exp\!\left[
-\frac{(y-a_i)^2}{2\sigma_\eta^2}
\right].
$

Substituting the two Gaussian conditional PDFs into the likelihood ratio gives:

$
\Lambda(y)
=
\frac{
\exp\!\left[
-\frac{(y-a_1)^2}{2\sigma_\eta^2}
\right]
}{
\exp\!\left[
-\frac{(y-a_0)^2}{2\sigma_\eta^2}
\right]
}.
$

The common normalization factor cancels. Taking the natural logarithm produces the log-likelihood ratio:

$
\ln\Lambda(y)
=
\frac{
(y-a_0)^2-(y-a_1)^2
}{
2\sigma_\eta^2
}.
$

Expanding the squared terms:

$
\ln\Lambda(y)
=
\frac{
2y(a_1-a_0)+a_0^2-a_1^2
}{
2\sigma_\eta^2
}.
$

Since:

$
a_0^2-a_1^2
=
-(a_1-a_0)(a_0+a_1),
$

the expression becomes:

$
\ln\Lambda(y)
=
\frac{a_1-a_0}{\sigma_\eta^2}
\left[
y-\frac{a_0+a_1}{2}
\right].
$

Because $a_1>a_0$, the log-likelihood ratio is an increasing function of $y$. The MAP test is therefore equivalent to comparing the received observation with a scalar threshold:

$
y
\mathop{\gtrless}_{H_0}^{H_1}
\gamma_{\mathrm{MAP}},
$

where:

$
\gamma_{\mathrm{MAP}}
=
\frac{a_0+a_1}{2}
+
\frac{\sigma_\eta^2}{a_1-a_0}
\ln\!\left(
\frac{P(H_0)}{P(H_1)}
\right).
$

The resulting decision rule is:

$
\widehat{H}(y)
=
\begin{cases}
H_0, & y<\gamma_{\mathrm{MAP}},\\[4pt]
H_1, & y\geq\gamma_{\mathrm{MAP}}.
\end{cases}
$

**Equal-prior special case**

If both hypotheses are equally likely:

$
P(H_0)=P(H_1),
$

then:

$
\ln\!\left(
\frac{P(H_0)}{P(H_1)}
\right)
=
0,
$

and the MAP threshold reduces to the midpoint between the two symbol amplitudes:

$
\gamma_{\mathrm{MAP}}
=
\frac{a_0+a_1}{2}.
$

For unequal priors, the threshold shifts toward the less probable hypothesis. This enlarges the decision region associated with the more probable symbol.

In [ ]:
# ============================================================
# Cell 8 — Analytical probability of error versus threshold
# ============================================================

if prior_h0 <= 0 or prior_h1 <= 0:
    raise ValueError("Both prior probabilities must be strictly positive.")

if not np.isclose(prior_h0 + prior_h1, 1.0):
    raise ValueError("The prior probabilities must sum to one.")

# ------------------------------------------------------------
# 1. Closed-form MAP threshold
# ------------------------------------------------------------

threshold_map = (
    (a0 + a1) / 2
    + (noise_variance / (a1 - a0))
    * np.log(prior_h0 / prior_h1)
)

# Ensure that the plotted threshold range includes the MAP threshold
threshold_lower_limit = min(lower_limit, threshold_map - noise_std)
threshold_upper_limit = max(upper_limit, threshold_map + noise_std)

threshold_values = np.linspace(
    threshold_lower_limit,
    threshold_upper_limit,
    1000,
)

# ------------------------------------------------------------
# 2. Conditional error probabilities
# ------------------------------------------------------------

false_alarm_probability = stats.norm.sf(
    threshold_values,
    loc=a0,
    scale=noise_std,
)

miss_probability = stats.norm.cdf(
    threshold_values,
    loc=a1,
    scale=noise_std,
)

# ------------------------------------------------------------
# 3. Total probability of error
# ------------------------------------------------------------

error_probability = (
    prior_h0 * false_alarm_probability
    + prior_h1 * miss_probability
)

minimum_error_index = np.argmin(error_probability)

threshold_numerical = threshold_values[minimum_error_index]
minimum_error_probability = error_probability[minimum_error_index]

# ------------------------------------------------------------
# 4. Error probability at the closed-form MAP threshold
# ------------------------------------------------------------

false_alarm_at_map = stats.norm.sf(
    threshold_map,
    loc=a0,
    scale=noise_std,
)

miss_at_map = stats.norm.cdf(
    threshold_map,
    loc=a1,
    scale=noise_std,
)

error_probability_at_map = (
    prior_h0 * false_alarm_at_map
    + prior_h1 * miss_at_map
)

# ------------------------------------------------------------
# 5. Visualization
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(12, 5),
    constrained_layout=True,
)

ax.plot(
    threshold_values,
    error_probability,
    linewidth=2.5,
    label=r"$P_e(\gamma)$",
)

ax.axvline(
    threshold_map,
    linestyle="--",
    linewidth=2.0,
    label=rf"MAP threshold: $\gamma_{{MAP}}={threshold_map:.3f}$",
)

ax.scatter(
    [threshold_map],
    [error_probability_at_map],
    s=70,
    zorder=3,
    label=rf"$P_e(\gamma_{{MAP}})={error_probability_at_map:.5f}$",
)

ax.set_title("Probability of binary-decision error versus threshold")
ax.set_xlabel(r"Decision threshold $\gamma$")
ax.set_ylabel(r"Probability of error $P_e(\gamma)$")
ax.set_xlim(threshold_lower_limit, threshold_upper_limit)
ax.legend()

plt.show()

# ------------------------------------------------------------
# 6. Numerical summary
# ------------------------------------------------------------

print(f"Numerically minimized threshold: {threshold_numerical:.6f}")
print(f"Closed-form MAP threshold:       {threshold_map:.6f}")
print(f"Grid minimum error probability:  {minimum_error_probability:.6f}")
print(f"MAP error probability:           {error_probability_at_map:.6f}")
print(f"False-alarm contribution:        {prior_h0 * false_alarm_at_map:.6f}")
print(f"Miss contribution:               {prior_h1 * miss_at_map:.6f}")

This cell  precomputes the Kernel Density Estimates (KDEs) for the noisy signals `a0` and `a1` over a comprehensive range of values (`X`).

In [ ]:
# ============================================================
# Cell 9 — Optional KDE-based numerical validation
# ============================================================

from scipy.integrate import cumulative_trapezoid

# ------------------------------------------------------------
# 1. Numerically integrate the KDE approximations
# ------------------------------------------------------------

cdf_h0_kde = cumulative_trapezoid(
    pdf_h0_kde,
    evaluation_points,
    initial=0.0,
)

cdf_h1_kde = cumulative_trapezoid(
    pdf_h1_kde,
    evaluation_points,
    initial=0.0,
)

if cdf_h0_kde[-1] <= 0 or cdf_h1_kde[-1] <= 0:
    raise ValueError(
        "The KDE-based CDF normalization failed. "
        "Check the KDE estimates or evaluation grid."
    )

# Normalize the finite-interval numerical CDFs
cdf_h0_kde /= cdf_h0_kde[-1]
cdf_h1_kde /= cdf_h1_kde[-1]

# ------------------------------------------------------------
# 2. Compute KDE-based error probabilities
# ------------------------------------------------------------

# False alarm:
# Decide H1 although H0 was transmitted.
false_alarm_probability_kde = 1.0 - cdf_h0_kde

# Miss:
# Decide H0 although H1 was transmitted.
miss_probability_kde = cdf_h1_kde

# Total KDE-based probability of error
error_probability_kde = (
    prior_h0 * false_alarm_probability_kde
    + prior_h1 * miss_probability_kde
)

# KDE-based threshold estimate
minimum_error_index_kde = np.argmin(error_probability_kde)

threshold_kde = evaluation_points[minimum_error_index_kde]
minimum_error_probability_kde = error_probability_kde[
    minimum_error_index_kde
]

# ------------------------------------------------------------
# 3. Compare analytical and KDE-based results
# ------------------------------------------------------------

plot_lower_limit = min(
    np.min(threshold_values),
    np.min(evaluation_points),
    threshold_map,
    threshold_kde,
)

plot_upper_limit = max(
    np.max(threshold_values),
    np.max(evaluation_points),
    threshold_map,
    threshold_kde,
)

fig, ax = plt.subplots(
    figsize=(12, 5),
    constrained_layout=True,
)

ax.plot(
    threshold_values,
    error_probability,
    linewidth=2.5,
    label="Analytical error probability",
)

ax.plot(
    evaluation_points,
    error_probability_kde,
    linestyle="--",
    linewidth=2.0,
    label="KDE-based numerical estimate",
)

ax.axvline(
    threshold_map,
    linestyle=":",
    linewidth=2.0,
    label=rf"Analytical MAP threshold: $\gamma_{{MAP}}={threshold_map:.3f}$",
)

ax.axvline(
    threshold_kde,
    linestyle="-.",
    linewidth=2.0,
    label=rf"KDE estimate: $\hat{{\gamma}}={threshold_kde:.3f}$",
)

ax.set_title(
    "Analytical and KDE-based estimates of the binary-decision error"
)

ax.set_xlabel(r"Decision threshold $\gamma$")
ax.set_ylabel(r"Probability of error $P_e(\gamma)$")
ax.set_xlim(plot_lower_limit, plot_upper_limit)
ax.legend()

plt.show()

# ------------------------------------------------------------
# 4. Numerical summary
# ------------------------------------------------------------

print(f"Analytical MAP threshold:        {threshold_map:.6f}")
print(f"KDE-based threshold estimate:    {threshold_kde:.6f}")
print(f"Analytical minimum error:        {error_probability_at_map:.6f}")
print(f"KDE-based minimum error:         {minimum_error_probability_kde:.6f}")
print(f"Absolute threshold difference:   {abs(threshold_kde - threshold_map):.6f}")


#### — Matched-filter detection for rectangular pulses

The previous sections considered a scalar detector with one noisy observation per transmitted symbol.

The sampled rectangular waveform contains $L$ samples per symbol:

$
x[n]
=
a_k,
\qquad
kL \leq n < (k+1)L,
$

where:

$
L
=
32.
$

Because the pulse shape is rectangular, the matched filter is equivalent to an integrate-and-dump receiver. For each symbol interval, the receiver combines the $L$ received samples.

Using the normalized block average:

$
Z_k
=
\frac{1}{L}
\sum_{\ell=0}^{L-1}
y[kL+\ell],
$

preserves the original symbol-amplitude scale. Under each hypothesis:

$
\begin{aligned}
H_0 &: \quad Z_k=a_0+\overline{\eta}_k,\\[4pt]
H_1 &: \quad Z_k=a_1+\overline{\eta}_k,
\end{aligned}
$

where:

$
\overline{\eta}_k
=
\frac{1}{L}
\sum_{\ell=0}^{L-1}
\eta[kL+\ell].
$

Since the AWGN samples are independent:

$
\overline{\eta}_k
\sim
\mathcal{N}
\left(
0,
\frac{\sigma_\eta^2}{L}
\right).
$

Therefore:

$
Z_k\mid H_0
\sim
\mathcal{N}
\left(
a_0,
\frac{\sigma_\eta^2}{L}
\right),
$

and:

$
Z_k\mid H_1
\sim
\mathcal{N}
\left(
a_1,
\frac{\sigma_\eta^2}{L}
\right).
$

The matched-filter MAP threshold is:

$
\gamma_{\mathrm{MF}}
=
\frac{a_0+a_1}{2}
+
\frac{\sigma_\eta^2/L}{a_1-a_0}
\ln
\left(
\frac{P(H_0)}{P(H_1)}
\right).
$

Compared with a single-sample detector, the matched filter reduces the effective noise variance by a factor of $L$. The corresponding signal-to-noise-ratio improvement is:

$
G_{\mathrm{MF}}
=
10\log_{10}(L)
\ \text{dB}.
$

For $L=32$:

$
G_{\mathrm{MF}}
\approx
15.05
\ \text{dB}.
$

This extension compares:

1. a detector that uses only one received sample per symbol interval;
2. a matched-filter detector that combines all $L$ samples in each interval.


In [ ]:
# ============================================================
# Cell 10 — Matched-filter detection for rectangular pulses
# ============================================================

# ------------------------------------------------------------
# 1. Validate and organize the received waveform
# ------------------------------------------------------------

a0, a1 = symbol_values
prior_h0, prior_h1 = symbol_probabilities

if a0 >= a1:
    raise ValueError("The binary amplitudes must satisfy a0 < a1.")

if prior_h0 <= 0 or prior_h1 <= 0:
    raise ValueError("Both prior probabilities must be strictly positive.")

if not np.isclose(prior_h0 + prior_h1, 1.0):
    raise ValueError("The prior probabilities must sum to one.")

if (
    not isinstance(samples_per_symbol, (int, np.integer))
    or samples_per_symbol <= 0
):
    raise ValueError("samples_per_symbol must be a positive integer.")

if noise_variance <= 0:
    raise ValueError("noise_variance must be strictly positive.")

num_transmitted_symbols = symbols.size
expected_num_samples = num_transmitted_symbols * samples_per_symbol

if received_waveform.size != expected_num_samples:
    raise ValueError(
        "The received waveform length must equal "
        "symbols.size * samples_per_symbol."
    )

received_blocks = received_waveform.reshape(
    num_transmitted_symbols,
    samples_per_symbol,
)

# ------------------------------------------------------------
# 2. Construct the two observation statistics
# ------------------------------------------------------------

# Baseline scalar detector:
# use one noisy waveform sample from each symbol interval.
scalar_sample_index = samples_per_symbol // 2

scalar_observations = received_blocks[:, scalar_sample_index]

# Matched-filter detector:
# for a rectangular pulse, the normalized integrate-and-dump
# statistic is the block average.
matched_filter_observations = np.mean(
    received_blocks,
    axis=1,
)

# ------------------------------------------------------------
# 3. Compute analytical MAP thresholds
# ------------------------------------------------------------

# Scalar detector: one observation per symbol.
threshold_scalar_map = (
    (a0 + a1) / 2
    + (noise_variance / (a1 - a0))
    * np.log(prior_h0 / prior_h1)
)

# Matched-filter detector:
# the block average reduces the effective noise variance by L.
matched_filter_noise_variance = (
    noise_variance / samples_per_symbol
)

matched_filter_noise_std = np.sqrt(
    matched_filter_noise_variance
)

threshold_matched_filter = (
    (a0 + a1) / 2
    + (matched_filter_noise_variance / (a1 - a0))
    * np.log(prior_h0 / prior_h1)
)

# ------------------------------------------------------------
# 4. Apply the decision rules
# ------------------------------------------------------------

scalar_decisions = np.where(
    scalar_observations >= threshold_scalar_map,
    a1,
    a0,
)

matched_filter_decisions = np.where(
    matched_filter_observations >= threshold_matched_filter,
    a1,
    a0,
)

# ------------------------------------------------------------
# 5. Count empirical errors
# ------------------------------------------------------------

scalar_error_mask = scalar_decisions != symbols

matched_filter_error_mask = (
    matched_filter_decisions != symbols
)

scalar_false_alarms = np.count_nonzero(
    (symbols == a0) & (scalar_decisions == a1)
)

scalar_misses = np.count_nonzero(
    (symbols == a1) & (scalar_decisions == a0)
)

matched_filter_false_alarms = np.count_nonzero(
    (symbols == a0) & (matched_filter_decisions == a1)
)

matched_filter_misses = np.count_nonzero(
    (symbols == a1) & (matched_filter_decisions == a0)
)

scalar_empirical_error_rate = np.mean(
    scalar_error_mask
)

matched_filter_empirical_error_rate = np.mean(
    matched_filter_error_mask
)

# ------------------------------------------------------------
# 6. Compute theoretical error probabilities
# ------------------------------------------------------------

scalar_false_alarm_probability = stats.norm.sf(
    threshold_scalar_map,
    loc=a0,
    scale=noise_std,
)

scalar_miss_probability = stats.norm.cdf(
    threshold_scalar_map,
    loc=a1,
    scale=noise_std,
)

scalar_theoretical_error_rate = (
    prior_h0 * scalar_false_alarm_probability
    + prior_h1 * scalar_miss_probability
)

matched_filter_false_alarm_probability = stats.norm.sf(
    threshold_matched_filter,
    loc=a0,
    scale=matched_filter_noise_std,
)

matched_filter_miss_probability = stats.norm.cdf(
    threshold_matched_filter,
    loc=a1,
    scale=matched_filter_noise_std,
)

matched_filter_theoretical_error_rate = (
    prior_h0 * matched_filter_false_alarm_probability
    + prior_h1 * matched_filter_miss_probability
)

# ------------------------------------------------------------
# 7. Exact matched-filter PDFs
# ------------------------------------------------------------

matched_filter_margin_std = 5.0

matched_filter_grid = np.linspace(
    min(a0, a1)
    - matched_filter_margin_std * matched_filter_noise_std,
    max(a0, a1)
    + matched_filter_margin_std * matched_filter_noise_std,
    1000,
)

matched_filter_pdf_h0 = stats.norm.pdf(
    matched_filter_grid,
    loc=a0,
    scale=matched_filter_noise_std,
)

matched_filter_pdf_h1 = stats.norm.pdf(
    matched_filter_grid,
    loc=a1,
    scale=matched_filter_noise_std,
)

weighted_matched_filter_pdf_h0 = (
    prior_h0 * matched_filter_pdf_h0
)

weighted_matched_filter_pdf_h1 = (
    prior_h1 * matched_filter_pdf_h1
)

# ------------------------------------------------------------
# 7b. Estimated matched-filter likelihoods from simulated data
# ------------------------------------------------------------

matched_filter_observations_h0 = matched_filter_observations[
    symbols == a0
]

matched_filter_observations_h1 = matched_filter_observations[
    symbols == a1
]

if matched_filter_observations_h0.size < 2:
    raise ValueError(
        "At least two matched-filter observations under H0 are required "
        "to estimate the KDE likelihood."
    )

if matched_filter_observations_h1.size < 2:
    raise ValueError(
        "At least two matched-filter observations under H1 are required "
        "to estimate the KDE likelihood."
    )

_, matched_filter_pdf_h0_kde = estimate_kde(
    samples=matched_filter_observations_h0,
    evaluation_points=matched_filter_grid,
    method="silverman",
)

_, matched_filter_pdf_h1_kde = estimate_kde(
    samples=matched_filter_observations_h1,
    evaluation_points=matched_filter_grid,
    method="silverman",
)

weighted_matched_filter_pdf_h0_kde = (
    prior_h0 * matched_filter_pdf_h0_kde
)

weighted_matched_filter_pdf_h1_kde = (
    prior_h1 * matched_filter_pdf_h1_kde
)
# ------------------------------------------------------------
# 8. Visualization
# ------------------------------------------------------------

num_visible_decisions = min(40, num_transmitted_symbols)
visible_symbol_indices = np.arange(num_visible_decisions)

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(12, 9),
    constrained_layout=True,
)

# Matched-filter statistics compared with transmitted symbols
axes[0].step(
    visible_symbol_indices,
    symbols[:num_visible_decisions],
    where="mid",
    linewidth=2.0,
    label="Transmitted symbols",
)

axes[0].scatter(
    visible_symbol_indices,
    matched_filter_observations[:num_visible_decisions],
    s=36,
    label="Matched-filter observations",
)

axes[0].axhline(
    threshold_matched_filter,
    linestyle="--",
    linewidth=2.0,
    label=(
        rf"Matched-filter threshold: "
        rf"$\gamma_{{\mathrm{{MF}}}}="
        rf"{threshold_matched_filter:.3f}$"
    ),
)

axes[0].set_title(
    "Matched-filter observations and transmitted symbols"
)

axes[0].set_xlabel("Symbol index")
axes[0].set_ylabel("Amplitude")
axes[0].legend()

# Prior-weighted matched-filter likelihoods
axes[1].plot(
    matched_filter_grid,
    weighted_matched_filter_pdf_h0,
    linewidth=2.5,
    label=r"Exact: $P(H_0)\,p(z\mid H_0)$",
)

axes[1].plot(
    matched_filter_grid,
    weighted_matched_filter_pdf_h1,
    linewidth=2.5,
    label=r"Exact: $P(H_1)\,p(z\mid H_1)$",
)

axes[1].plot(
    matched_filter_grid,
    weighted_matched_filter_pdf_h0_kde,
    linestyle="--",
    linewidth=2.0,
    label=r"Estimated: $P(H_0)\,\hat{p}(z\mid H_0)$",
)

axes[1].plot(
    matched_filter_grid,
    weighted_matched_filter_pdf_h1_kde,
    linestyle="--",
    linewidth=2.0,
    label=r"Estimated: $P(H_1)\,\hat{p}(z\mid H_1)$",
)

axes[1].axvline(
    threshold_matched_filter,
    linestyle=":",
    linewidth=2.2,
    label=(
        rf"$\gamma_{{\mathrm{{MF}}}}="
        rf"{threshold_matched_filter:.3f}$"
    ),
)

axes[1].set_title(
    "Exact and estimated prior-weighted likelihoods after matched filtering"
)

axes[1].set_xlabel(
    r"Matched-filter statistic $z$"
)

axes[1].set_ylabel(
    "Weighted probability density"
)

axes[1].set_xlim(
    np.min(matched_filter_grid),
    np.max(matched_filter_grid),
)
axes[1].legend()

plt.show()

# ------------------------------------------------------------
# 9. Numerical summary
# ------------------------------------------------------------

matched_filter_gain_db = (
    10 * np.log10(samples_per_symbol)
)

print("Scalar detector")
print("----------------")
print(
    f"MAP threshold:                "
    f"{threshold_scalar_map:.6f}"
)
print(
    f"Empirical symbol-error rate:  "
    f"{scalar_empirical_error_rate:.6e}"
)
print(
    f"Theoretical error rate:       "
    f"{scalar_theoretical_error_rate:.6e}"
)
print(
    f"False alarms:                 "
    f"{scalar_false_alarms}"
)
print(
    f"Misses:                       "
    f"{scalar_misses}"
)

print()

print("Matched-filter detector")
print("-----------------------")
print(
    f"Samples per symbol:           "
    f"{samples_per_symbol}"
)
print(
    f"Effective noise variance:     "
    f"{matched_filter_noise_variance:.6f}"
)
print(
    f"MAP threshold:                "
    f"{threshold_matched_filter:.6f}"
)
print(
    f"Empirical symbol-error rate:  "
    f"{matched_filter_empirical_error_rate:.6e}"
)
print(
    f"Theoretical error rate:       "
    f"{matched_filter_theoretical_error_rate:.6e}"
)
print(
    f"False alarms:                 "
    f"{matched_filter_false_alarms}"
)
print(
    f"Misses:                       "
    f"{matched_filter_misses}"
)
print(
    f"Matched-filter gain:          "
    f"{matched_filter_gain_db:.2f} dB"
)

### Binary Detector using Multilayer Perceptron (MLP)

The MLP will learn to classify the received noisy symbol blocks (`received_blocks`) into the original transmitted symbols (`symbols`).

In [ ]:
# ============================================================
# Cell 11 — MLP data preparation for binary detection
# ============================================================

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

mlp_seed = 42

# ------------------------------------------------------------
# 2. Prepare supervised-learning data
# ------------------------------------------------------------

# Use the matched-filter statistic as the MLP input.
# Each row contains one scalar sufficient statistic z_k.
X = matched_filter_observations.reshape(-1, 1).astype(np.float32)

# Binary labels:
# map a0 -> 0 and a1 -> 1.
y_encoded = np.where(symbols == a1, 1, 0).astype(int)

# Keep sample indices so that classical detectors and MLP
# are evaluated on the same test subset.
sample_ids = np.arange(symbols.size)

# ------------------------------------------------------------
# 3. Consistency checks
# ------------------------------------------------------------

if X.ndim != 2 or X.shape[1] != 1:
    raise ValueError("X must have shape (num_symbols, 1).")

if X.shape[0] != symbols.size:
    raise ValueError("The number of MLP inputs must equal the number of symbols.")

if y_encoded.shape[0] != symbols.size:
    raise ValueError("The number of labels must equal the number of symbols.")

if np.unique(y_encoded).size != 2:
    raise ValueError("Both binary classes must be present in the dataset.")

# ------------------------------------------------------------
# 4. Stratified train-test split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X,
    y_encoded,
    sample_ids,
    test_size=0.2,
    random_state=mlp_seed,
    stratify=y_encoded,
)

# ------------------------------------------------------------
# 5. Numerical summary
# ------------------------------------------------------------

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test:  {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test:  {y_test.shape}")

print()
print(f"Training P(H0):   {np.mean(y_train == 0):.4f}")
print(f"Training P(H1):   {np.mean(y_train == 1):.4f}")
print(f"Testing P(H0):    {np.mean(y_test == 0):.4f}")
print(f"Testing P(H1):    {np.mean(y_test == 1):.4f}")

print()
print(f"idx_train size:   {idx_train.size}")
print(f"idx_test size:    {idx_test.size}")

In [ ]:
# ============================================================
# Cell 12 — MLP model definition and training
# ============================================================

try:
    import tensorflow as tf
except ImportError as exc:
    raise ImportError(
        "TensorFlow is required for the optional MLP section. "
        "Install it with `pip install tensorflow`, or skip Cells 11–13."
    ) from exc


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

mlp_seed = 42
tf.keras.utils.set_random_seed(mlp_seed)
tf.config.experimental.enable_op_determinism()

# ------------------------------------------------------------
# 2. Define a compact MLP binary detector
# ------------------------------------------------------------

model = Sequential(
    [
        Input(shape=(1,)),
        Dense(8, activation="relu"),
        Dense(4, activation="relu"),
        Dense(1, activation="sigmoid"),
    ]
)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# ------------------------------------------------------------
# 3. Train the MLP
# ------------------------------------------------------------

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.1,
    verbose=1,
)

model.summary()

# ------------------------------------------------------------
# 4. Plot training history
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(12, 5),
    constrained_layout=True,
)

axes[0].plot(
    history.history["accuracy"],
    label="Training accuracy",
)

axes[0].plot(
    history.history["val_accuracy"],
    label="Validation accuracy",
)

axes[0].set_title("MLP accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(
    history.history["loss"],
    label="Training loss",
)

axes[1].plot(
    history.history["val_loss"],
    label="Validation loss",
)

axes[1].set_title("MLP loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Binary cross-entropy")
axes[1].legend()

plt.show()

In [ ]:
# ============================================================
# Cell 13 — MLP vs. classical detection theory comparison
# ============================================================

# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

if "model" not in globals():
    raise RuntimeError("Train the MLP in Cell 12 first.")

if "idx_test" not in globals():
    raise RuntimeError(
        "The test indices idx_test are required for a fair comparison. "
        "Modify the MLP data-preparation cell so that train_test_split "
        "also returns idx_train and idx_test."
    )

# ------------------------------------------------------------
# 2. MLP predictions on the test subset
# ------------------------------------------------------------

X_test_mlp = matched_filter_observations[idx_test].reshape(-1, 1).astype(np.float32)

mlp_proba_test = model.predict(
    X_test_mlp,
    verbose=0,
).reshape(-1)

mlp_pred_labels_test = (mlp_proba_test >= 0.5).astype(int)
mlp_pred_symbols_test = np.where(
    mlp_pred_labels_test == 1,
    a1,
    a0,
)

symbols_test = symbols[idx_test]

mlp_error_mask_test = mlp_pred_symbols_test != symbols_test

mlp_empirical_error_rate_test = np.mean(mlp_error_mask_test)

mlp_false_alarms_test = np.count_nonzero(
    (symbols_test == a0) & (mlp_pred_symbols_test == a1)
)

mlp_misses_test = np.count_nonzero(
    (symbols_test == a1) & (mlp_pred_symbols_test == a0)
)

# ------------------------------------------------------------
# 3. Classical detector performance on the same test subset
# ------------------------------------------------------------

scalar_decisions_test = scalar_decisions[idx_test]
matched_filter_decisions_test = matched_filter_decisions[idx_test]

scalar_error_mask_test = scalar_decisions_test != symbols_test
matched_filter_error_mask_test = matched_filter_decisions_test != symbols_test

scalar_empirical_error_rate_test = np.mean(
    scalar_error_mask_test
)

matched_filter_empirical_error_rate_test = np.mean(
    matched_filter_error_mask_test
)

scalar_false_alarms_test = np.count_nonzero(
    (symbols_test == a0) & (scalar_decisions_test == a1)
)

scalar_misses_test = np.count_nonzero(
    (symbols_test == a1) & (scalar_decisions_test == a0)
)

matched_filter_false_alarms_test = np.count_nonzero(
    (symbols_test == a0) & (matched_filter_decisions_test == a1)
)

matched_filter_misses_test = np.count_nonzero(
    (symbols_test == a1) & (matched_filter_decisions_test == a0)
)

# ------------------------------------------------------------
# 4. Tabular comparison
# ------------------------------------------------------------

print("=" * 78)
print("DETECTOR PERFORMANCE COMPARISON")
print("=" * 78)
print(
    f"{'Detector':<34} "
    f"{'Error Rate':<16} "
    f"{'False Alarms':<14} "
    f"{'Misses':<8}"
)
print("-" * 78)

print(
    f"{'Scalar MAP theory':<34} "
    f"{scalar_theoretical_error_rate:<16.6e} "
    f"{'—':<14} "
    f"{'—':<8}"
)

print(
    f"{'Matched-filter MAP theory':<34} "
    f"{matched_filter_theoretical_error_rate:<16.6e} "
    f"{'—':<14} "
    f"{'—':<8}"
)

print(
    f"{'Scalar empirical, test set':<34} "
    f"{scalar_empirical_error_rate_test:<16.6e} "
    f"{scalar_false_alarms_test:<14} "
    f"{scalar_misses_test:<8}"
)

print(
    f"{'Matched-filter empirical, test set':<34} "
    f"{matched_filter_empirical_error_rate_test:<16.6e} "
    f"{matched_filter_false_alarms_test:<14} "
    f"{matched_filter_misses_test:<8}"
)

print(
    f"{'MLP empirical, test set':<34} "
    f"{mlp_empirical_error_rate_test:<16.6e} "
    f"{mlp_false_alarms_test:<14} "
    f"{mlp_misses_test:<8}"
)

print("=" * 78)

# ------------------------------------------------------------
# 5. Visualization
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(14, 5.5),
    constrained_layout=True,
)

# Left: error-rate comparison
detectors = [
    "Scalar MAP\n(theory)",
    "MF MAP\n(theory)",
    "Scalar\n(test)",
    "MF\n(test)",
    "MLP\n(test)",
]

error_rates = np.array(
    [
        scalar_theoretical_error_rate,
        matched_filter_theoretical_error_rate,
        scalar_empirical_error_rate_test,
        matched_filter_empirical_error_rate_test,
        mlp_empirical_error_rate_test,
    ],
    dtype=float,
)

# Avoid log-scale failure when an empirical error rate is zero.
plot_error_rates = np.maximum(error_rates, 1e-12)

bars = axes[0].bar(
    detectors,
    plot_error_rates,
    alpha=0.75,
    edgecolor="black",
)

axes[0].set_ylabel("Probability of error")
axes[0].set_title("Analytical and test-set empirical error rates")
axes[0].set_yscale("log")
axes[0].grid(
    True,
    which="both",
    axis="y",
    linestyle="--",
    alpha=0.5,
)

for bar, rate in zip(bars, error_rates):
    label = "0" if rate == 0 else f"{rate:.2e}"
    axes[0].annotate(
        label,
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

# Right: learned MLP probability as a function of the sufficient statistic
z_test = matched_filter_observations[idx_test]

axes[1].scatter(
    z_test,
    mlp_proba_test,
    c=y_test,
    alpha=0.75,
    edgecolors="black",
    linewidth=0.5,
    label="Test samples",
)

z_grid = np.linspace(
    np.min(matched_filter_observations),
    np.max(matched_filter_observations),
    400,
).reshape(-1, 1).astype(np.float32)

mlp_proba_grid = model.predict(
    z_grid,
    verbose=0,
).reshape(-1)

axes[1].plot(
    z_grid.reshape(-1),
    mlp_proba_grid,
    linewidth=2.5,
    label=r"MLP learned $P(\hat{a}=a_1\mid z)$",
)

axes[1].axvline(
    threshold_matched_filter,
    linestyle="--",
    linewidth=2.0,
    label=rf"MF MAP threshold: {threshold_matched_filter:.3f}",
)

axes[1].axhline(
    0.5,
    linestyle="-",
    linewidth=1.0,
    alpha=0.5,
    label="MLP decision boundary",
)

axes[1].set_xlabel(r"Matched-filter statistic $z$")
axes[1].set_ylabel(r"MLP predicted probability $P(\hat{a}=a_1)$")
axes[1].set_title("MLP decision curve in sufficient-statistic space")
axes[1].set_ylim(-0.05, 1.05)
axes[1].legend(loc="best", fontsize=8.5)

plt.show()

# ------------------------------------------------------------
# 6. Effective MLP threshold from the learned curve
# ------------------------------------------------------------

crossing_mask = np.diff(np.sign(mlp_proba_grid - 0.5)) != 0

if np.any(crossing_mask):
    crossing_idx = np.where(crossing_mask)[0][0]

    z_left = z_grid[crossing_idx, 0]
    z_right = z_grid[crossing_idx + 1, 0]

    p_left = mlp_proba_grid[crossing_idx]
    p_right = mlp_proba_grid[crossing_idx + 1]

    effective_mlp_threshold = z_left + (0.5 - p_left) * (
        z_right - z_left
    ) / (p_right - p_left)

    print()
    print(
        f"Effective MLP threshold:       "
        f"{effective_mlp_threshold:.6f}"
    )
    print(
        f"Analytical MF MAP threshold:   "
        f"{threshold_matched_filter:.6f}"
    )
    print(
        f"Absolute difference:           "
        f"{abs(effective_mlp_threshold - threshold_matched_filter):.6f}"
    )

else:
    print()
    print(
        "No clear 0.5 crossing was found in the learned MLP decision curve."
    )
